# Rigid-Bar Network: Full Diagnostic Analysis

Analyzes output from `rigid_bar_network_2d_straintest.c` across all its modes:
`nve`, `gmod`, `straintest`, `equiltest`, `msdtest`, `connectivity`, `snapshot`.

Set `base` below to match your network filename (without `.dat`), and adjust
the specific filenames in each section as needed -- not every run produces
every file, so only run the cells for the outputs you actually generated.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import re

# Adjust to match your network file (without .dat)
base = "floppy_network_N1600_z3.600_s00000"


## 1. NVE diagnostics

From: `./rigid_bar_straintest network.dat nve [T] [dt] [steps] [out_every]`
Output: `<network>_nve.dat`

In [ ]:
def load_nve(path):
    return pd.read_csv(path, sep='\t', comment='#',
        names=['step','KE','bar_err','vel_constraint','sigma_xy','pressure','px','py'])

nve = load_nve(f"{base}_nve.dat")

fig, axes = plt.subplots(2, 2, figsize=(11, 7))

axes[0,0].plot(nve['step'], nve['KE'])
axes[0,0].set_xlabel('step'); axes[0,0].set_ylabel('KE')
axes[0,0].set_title('Kinetic energy (should be flat -- energy conservation check)')

axes[0,1].plot(nve['step'], nve['bar_err'], color='crimson')
axes[0,1].set_yscale('log')
axes[0,1].axhline(1e-8, color='k', ls='--', lw=1)
axes[0,1].set_xlabel('step'); axes[0,1].set_ylabel('max bar-length error')
axes[0,1].set_title('Constraint drift (should stay pinned near 1e-8)')

axes[1,0].plot(nve['step'], nve['sigma_xy'], color='tab:blue')
axes[1,0].set_xlabel('step'); axes[1,0].set_ylabel(r'$\sigma_{xy}$ ($=\Gamma$)')
axes[1,0].set_title('Instantaneous stress (purely kinetic -- expect noisy fluctuation)')

axes[1,1].plot(nve['step'], np.sqrt(nve['px']**2 + nve['py']**2), color='tab:green')
axes[1,1].set_xlabel('step'); axes[1,1].set_ylabel('|momentum|')
axes[1,1].set_title('Total momentum (should stay near zero)')

plt.tight_layout()
plt.show()


## 2. Linear-response shear modulus G(z)

From: `./rigid_bar_straintest network.dat gmod [dgamma_list] [T] [dt] [dyn_steps]`
Output: appended (across ALL networks/z you've run gmod on) to `G_curvature_vs_z.dat`
in the working directory -- this file is NOT per-network, it accumulates.

In [ ]:
def load_gmod(path="G_curvature_vs_z.dat"):
    # This file has a genuine header row (no '#' prefix), unlike the other
    # outputs -- read it directly rather than overriding column names.
    return pd.read_csv(path, sep='\t')

gmod = load_gmod()

fig, ax = plt.subplots(figsize=(6.5,4.5))
for dg, g in gmod.groupby('dgamma'):
    g_sorted = g.sort_values('z')
    ax.plot(g_sorted['z'], g_sorted['G_fd'], 'o-', label=f'dgamma={dg:.0e}')
ax.set_xlabel('z')
ax.set_ylabel(r'$G_{fd}$')
ax.set_title('Finite-difference shear modulus vs coordination number')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


## 3. Strain-ramp test

From: `./rigid_bar_straintest network.dat straintest [n_strain_steps] [dgamma] [enable_correction] [max_corrections] [corr_tol] [dyn_steps] [sample_every] [lambda_sample_every] [power_iters] [T] [dt]`

Outputs: `<network>_straintest_summary.dat`, `<network>_straintest_corrtrace.dat`

If you ran both `enable_correction=1` and `enable_correction=0` versions, save
them to different filenames (e.g. `_on_` / `_off_`) and load both below.

In [ ]:
def load_straintest_summary(path):
    return pd.read_csv(path, sep='\t', comment='#',
        names=['strain_step','cum_strain','n_corr','converged','bar_err_corr',
               'sigma_preCorrection','sigma_postCorrection','sigma_endpoint',
               'sigma_mean','sigma_stderr','n_sigma_samples',
               'lambda_preCorrection','lambda_postCorrection',
               'lambda_mean','lambda_stderr','n_lambda_samples','n_cg_failures',
               'pressure_postdyn','bar_err_postdyn','vel_con_postdyn','KE_postdyn'])

def load_straintest_corrtrace(path):
    return pd.read_csv(path, sep='\t', comment='#',
        names=['strain_step','cum_strain','corr_iter','max_bar_err'])

summary = load_straintest_summary(f"{base}_straintest_summary.dat")
trace   = load_straintest_corrtrace(f"{base}_straintest_corrtrace.dat")

# flag strain steps where something went numerically suspect
suspect = (summary['converged'] == 0) | (summary['bar_err_postdyn'] > 1e-6) | (summary['n_cg_failures'] > 0)

summary[['strain_step','n_corr','converged','bar_err_corr','bar_err_postdyn',
         'n_cg_failures','lambda_mean','sigma_mean']].assign(suspect=suspect)


In [ ]:
# 3a. Correction convergence: is one pass enough?
fig, ax = plt.subplots(figsize=(6.5,4.5))
for step, g in trace.groupby('strain_step'):
    ax.plot(g['corr_iter'], g['max_bar_err'], 'o-', label=f"step {step}")
ax.axhline(1e-8, color='k', ls='--', lw=1, label='corr_tol')
ax.set_yscale('log')
ax.set_xlabel('correction pass number')
ax.set_ylabel('max relative bar-length error')
ax.set_title('Does one correction suffice after imposing strain?')
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()


In [ ]:
# 3b. Number of corrections used per strain step
fig, ax = plt.subplots(figsize=(6.5,4))
ax.step(summary['strain_step'], summary['n_corr'], where='mid', marker='o')
ax.scatter(summary.loc[suspect,'strain_step'], summary.loc[suspect,'n_corr'],
           color='red', zorder=5, label='suspect step')
ax.set_xlabel('strain step')
ax.set_ylabel('# corrections used to reach corr_tol')
ax.set_title('Correction count vs strain step')
if suspect.any(): ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# 3c. Stress vs accumulated strain (single endpoint snapshot -- the raw signal)
fig, ax = plt.subplots(figsize=(6.5,4.5))
ax.plot(summary['cum_strain'], summary['sigma_endpoint'], 's-')
ax.plot(summary.loc[suspect,'cum_strain'], summary.loc[suspect,'sigma_endpoint'],
        'rx', markersize=10, label='suspect (CG failure / bad convergence)')
ax.set_xlabel(r'cumulative strain $\gamma$')
ax.set_ylabel(r'$\sigma_{xy}$')
ax.set_title('Stress vs accumulated strain (endpoint snapshot)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# 3d. Time-averaged stress vs strain (noise-reduced version)
fig, ax = plt.subplots(figsize=(6.5,4.5))
ax.errorbar(summary['cum_strain'], summary['sigma_mean'], yerr=summary['sigma_stderr'],
            fmt='s-', capsize=2)
ax.plot(summary.loc[suspect,'cum_strain'], summary.loc[suspect,'sigma_mean'],
        'rx', markersize=10, label='suspect')
ax.set_xlabel(r'cumulative strain $\gamma$')
ax.set_ylabel(r'$\sigma_{xy}$ (time-averaged $\pm$ stderr)')
ax.set_title('Time-averaged stress vs accumulated strain')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# 3e. Does the correction step itself change the measured stress?
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7,7), sharex=True,
                                gridspec_kw={'height_ratios':[3,1]})
ax1.plot(summary['cum_strain'], summary['sigma_preCorrection'], 'o--',
         color='tab:blue', label=r'$\sigma_{xy}$ no correction (raw shear)')
ax1.plot(summary['cum_strain'], summary['sigma_postCorrection'], 's-',
         color='tab:orange', label=r'$\sigma_{xy}$ with correction')
ax1.set_ylabel(r'$\sigma_{xy}$')
ax1.set_title('Effect of the correction step on measured stress')
ax1.legend()
ax2.step(summary['cum_strain'], summary['n_corr'], where='mid', color='k')
ax2.set_ylabel('# corrections'); ax2.set_xlabel(r'cumulative strain $\gamma$')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6.5,4))
shift = summary['sigma_postCorrection'] - summary['sigma_preCorrection']
ax.plot(summary['cum_strain'], shift, 'o-', color='purple')
ax.axhline(0, color='k', lw=0.8)
ax.set_xlabel(r'cumulative strain $\gamma$')
ax.set_ylabel(r'$\sigma_{postCorrection}-\sigma_{preCorrection}$')
ax.set_title('Correction-induced shift in stress')
plt.tight_layout()
plt.show()


In [ ]:
# 3f. lambda_min(N) alongside stress -- and its own pre/post-correction pair
fig, ax1 = plt.subplots(figsize=(7,4.5))
ax1.plot(summary['strain_step'], summary['sigma_mean'], 'o-', color='tab:blue')
ax1.set_ylabel(r'$\sigma_{xy}$ (mean)', color='tab:blue')
ax1.set_xlabel('strain step')
ax2 = ax1.twinx()
ax2.plot(summary['strain_step'], summary['lambda_mean'], 's-', color='tab:green', alpha=0.7)
ax2.set_ylabel(r'$\lambda_{min}(N)$ (mean)', color='tab:green')
ax1.set_title('Stress and conditioning together')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6.5,4))
ax.plot(summary['cum_strain'], summary['lambda_preCorrection'], 'o--', label='pre-correction')
ax.plot(summary['cum_strain'], summary['lambda_postCorrection'], 's-', label='post-correction')
ax.set_xlabel(r'cumulative strain $\gamma$'); ax.set_ylabel(r'$\lambda_{min}(N)$')
ax.set_title('Does correction change the conditioning of N?')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 3g. CG failures per strain step -- direct test of the conditioning hypothesis
fig, ax = plt.subplots(figsize=(6.5,4))
ax.bar(summary['strain_step'], summary['n_cg_failures'], color='crimson')
ax.set_xlabel('strain step'); ax.set_ylabel('# CG failures in window')
ax.set_title('Direct count of non-converged CG solves per strain step')
plt.tight_layout()
plt.show()


**Optional**: if you saved both an `enable_correction=1` run and an
`enable_correction=0` run (same `dgamma`, `dyn_steps`, etc.), load both and
compare directly:

In [ ]:
# summary_on  = load_straintest_summary(f"{base}_on_straintest_summary.dat")
# summary_off = load_straintest_summary(f"{base}_off_straintest_summary.dat")
# fig, ax = plt.subplots(figsize=(7,4.5))
# ax.plot(summary_off['strain_step'], summary_off['bar_err_postdyn'], 'o--', label='no correction')
# ax.plot(summary_on['strain_step'],  summary_on['bar_err_postdyn'],  's-',  label='with correction')
# ax.set_yscale('log'); ax.set_xlabel('strain step'); ax.set_ylabel('bar error (post-dynamics)')
# ax.legend(); plt.tight_layout(); plt.show()


## 4. Fixed-strain equilibration test: does $\lambda_{min}(N)$ settle?

From: `./rigid_bar_straintest network.dat equiltest [N_steps] [sample_every] [power_iters] [snapshot_every] [T] [dt]`
Output: `<network>_equiltest.dat` (plus snapshot files if `snapshot_every>0`, used in Section 6/7).

Run this once per network/z of interest and give each output file a distinct
name (or keep them in per-z subfolders) so you can compare transient lengths
across z, as your PI asked.

In [ ]:
def load_equiltest(path):
    return pd.read_csv(path, sep='\t', comment='#',
        names=['step','lambda_min','gamma','bar_err','KE','n_cg_failures'])

# List every *_equiltest.dat you have; label by whatever distinguishes them (e.g. z)
equiltest_files = sorted(glob.glob("*_equiltest.dat"))
print("Found:", equiltest_files)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7.5,6.5), sharex=True)
for fpath in equiltest_files:
    eq = load_equiltest(fpath)
    label = fpath.replace('_equiltest.dat', '')
    ax1.plot(eq['step'], eq['lambda_min'], 'o-', markersize=3, label=label)
    ax2.plot(eq['step'], eq['bar_err'], alpha=0.7, label=label)

ax1.set_ylabel(r'$\lambda_{min}(N)$')
ax1.set_title('Configurational equilibration: does $\lambda_{min}(N)$ stop drifting?')
ax1.legend(fontsize=7)
ax2.set_yscale('log')
ax2.set_ylabel('bar error'); ax2.set_xlabel('RATTLE step')
plt.tight_layout()
plt.show()


## 5. Mean-squared displacement

From: `./rigid_bar_straintest network.dat msdtest [equil_steps] [msd_steps] [sample_every] [T] [dt]`
Output: `<network>_msdtest.dat`

In [ ]:
def load_msdtest(path):
    return pd.read_csv(path, sep='\t', comment='#',
        names=['step','msd','msd_com_subtracted','bar_err','KE'])

msd = load_msdtest(f"{base}_msdtest.dat")

fig, ax = plt.subplots(figsize=(6.5,4.5))
ax.plot(msd['step'], msd['msd'], 'o-', label='raw MSD')
ax.plot(msd['step'], msd['msd_com_subtracted'], 's--', label='COM-subtracted', alpha=0.7)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('RATTLE steps since equilibration')
ax.set_ylabel(r'$\langle[r(t)-r(0)]^2\rangle$')
ax.set_title('Mean-squared displacement')
ax.legend()
plt.tight_layout()
plt.show()

# Reference slopes: MSD ~ t^1 (diffusive) and ~ t^2 (ballistic), for comparison
fig, ax = plt.subplots(figsize=(6.5,4.5))
mask = msd['step'] > 0
ax.loglog(msd.loc[mask,'step'], msd.loc[mask,'msd'], 'o-', label='MSD')
t_ref = msd.loc[mask,'step'].values
scale = msd.loc[mask,'msd'].values[0] / t_ref[0]
ax.loglog(t_ref, scale*t_ref, 'k--', alpha=0.5, label=r'$\propto t^1$ (diffusive)')
scale2 = msd.loc[mask,'msd'].values[0] / t_ref[0]**2
ax.loglog(t_ref, scale2*t_ref**2, 'k:', alpha=0.5, label=r'$\propto t^2$ (ballistic)')
ax.set_xlabel('step'); ax.set_ylabel('MSD'); ax.legend(fontsize=8)
ax.set_title('MSD scaling regime')
plt.tight_layout()
plt.show()


## 6. $\lambda_{min}(N)$ vs time across z, from saved snapshots (batch/offline)

Generate snapshots with `equiltest`'s `snapshot_every` option (Section 4), then
analyze each one independently -- cheap, parallelizable, and works across as
many z's as you like without keeping any single run alive:

```bash
for f in *_stepXXXXXXXX.dat; do ./rigid_bar_straintest "$f" snapshot 30; done > lambda_snapshots.tsv
```

The `snapshot` mode's stdout mixes an unrelated `Network: ...` line (printed
by `readNetwork` on every invocation) in with the actual data line -- the
parser below only keeps lines containing `lambda_min(N)=`.

In [ ]:
def parse_snapshot_line(line):
    if 'lambda_min(N)=' not in line:
        return None
    fields = line.strip().split('\t')
    fname = fields[0]
    d = {'file': fname}
    for f in fields[1:]:
        key, val = f.split('=')
        d[key.replace('(N)', '_N')] = float(val)
    return d

def load_snapshot_tsv(path):
    rows = []
    with open(path) as f:
        for line in f:
            r = parse_snapshot_line(line)
            if r is not None:
                rows.append(r)
    df = pd.DataFrame(rows)
    # extract the step number embedded in the filename, e.g. ..._step00012000.dat
    df['step'] = df['file'].apply(lambda s: int(re.search(r'_step(\d+)\.dat', s).group(1))
                                   if re.search(r'_step(\d+)\.dat', s) else np.nan)
    return df.sort_values('step')

snap = load_snapshot_tsv("lambda_snapshots.tsv")
snap[['file','step','N','z','strain','lambda_min_N','bar_err']]


In [ ]:
# If lambda_snapshots.tsv contains snapshots from multiple z (mixed together
# because you concatenated several networks' outputs), split and plot by z:
fig, ax = plt.subplots(figsize=(7,4.5))
for zval, g in snap.groupby('z'):
    g_sorted = g.sort_values('step')
    ax.plot(g_sorted['step'], g_sorted['lambda_min_N'], 'o-', label=f'z={zval:.3g}')
ax.set_xlabel('step'); ax.set_ylabel(r'$\lambda_{min}(N)$')
ax.set_title(r'$\lambda_{min}(N)$ vs time, from saved snapshots, by $z$')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


## 7. Connectivity check (loose islands)

From: `./rigid_bar_straintest network.dat connectivity` -- prints directly to
stdout (component count, largest component size, isolated-node count); no
file is written. Run this on your candidate networks from lowest z upward and
use the lowest one reporting "spans the whole network -- no loose islands"
before recording a movie in Section 8.

## 8. Movie of the dynamics, from saved snapshots

Uses the `.dat` snapshot files produced by `equiltest`'s `snapshot_every`
option (Section 4/6). Bonds are drawn with the minimum-image convention (same
as the C code's own `updateGeometry`), so a bond that wraps around the box
edge is drawn as a short segment toward the correct neighbor image rather
than one spurious line across the box.

In [ ]:
from matplotlib.animation import FuncAnimation

def load_snapshot_network(path):
    with open(path) as f:
        lines = f.readlines()
    L, strain_val = float(lines[0].split()[0]), float(lines[0].split()[1])
    rx, ry = [], []
    i = 1
    while True:
        parts = lines[i].split()
        a, b, c = float(parts[0]), float(parts[1]), float(parts[2])
        # the "contacts" header line looks like "<int contacts>\t-1\t-1"
        if c == -1.0 and b == -1.0 and a == int(a) and a > 1.5:
            contacts = int(a)
            i += 1
            break
        rx.append(a); ry.append(b)
        i += 1
    bonds = []
    for l in range(contacts):
        parts = lines[i+l].split()
        bonds.append((int(parts[0]), int(parts[1])))
    return L, strain_val, np.array(rx), np.array(ry), bonds

def pbc_segment(xi, yi, xj, yj):
    dx, dy = xj - xi, yj - yi
    dx -= round(dx); dy -= round(dy)   # minimum image, reduced coords in [0,1)
    return (xi, yi), (xi + dx, yi + dy)

snapshot_files = sorted(glob.glob(f"{base}_step*.dat"))
print(f"Found {len(snapshot_files)} snapshots")

fig, ax = plt.subplots(figsize=(6,6))

def render_frame(idx):
    ax.clear()
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect('equal')
    L, strain_val, rx, ry, bonds = load_snapshot_network(snapshot_files[idx])
    for i, j in bonds:
        (x0, y0), (x1, y1) = pbc_segment(rx[i], ry[i], rx[j], ry[j])
        ax.plot([x0, x1], [y0, y1], color='steelblue', lw=0.7, alpha=0.7)
    ax.scatter(rx, ry, s=8, color='crimson', zorder=3)
    step_label = re.search(r'_step(\d+)\.dat', snapshot_files[idx]).group(1)
    ax.set_title(f"step {int(step_label)}  (strain={strain_val:.4g})")

if snapshot_files:
    anim = FuncAnimation(fig, render_frame, frames=len(snapshot_files), interval=150)
    anim.save(f"{base}_dynamics.mp4", writer='ffmpeg', dpi=150)
    # anim.save(f"{base}_dynamics.gif", writer='pillow', dpi=100)  # fallback if ffmpeg unavailable
    print(f"Saved {base}_dynamics.mp4")
else:
    print("No snapshot files found -- run equiltest with snapshot_every > 0 first.")
